# Proses pelatihan AST (Audio Spectrogram Transformer)

**Transformer yang bekerja pada gambar suara, dipra-latih untuk mengenali suara sehari hari pada AudioSet: gonggongan, pintu, musik, mesin.**

Notebook ini sengaja dipecah menjadi delapan langkah terpisah, satu sel per
langkah, supaya prosesnya bisa dijelaskan sambil berjalan. Notebook ini bukan
untuk mengejar hasil secepat mungkin.

| langkah | isi | perkiraan waktu |
|---|---|---|
| 1 | Menyiapkan runtime, kode, dan dependensi | 1 sampai 2 menit |
| 2 | Mengambil dan memverifikasi dataset | 1 menit dari cache, 10 menit dari arsip |
| 3 | Mendengarkan dan melihat datanya | seketika |
| 4 | Apa yang sebenarnya dilihat model ini | 1 sampai 2 menit |
| 5 | Bentuk modelnya dan berapa yang dilatih | seketika |
| 6 | Bagaimana data sengaja dirusak, dan kenapa | seketika |
| 7 | Melatih | lihat catatan di langkah 7 |
| 8 | Membaca hasil dan kesalahannya | seketika |

**Sebelum mulai:** menu `Runtime` lalu `Change runtime type`, pilih **T4 GPU**,
lalu `Save`. Tanpa itu langkah 1 akan berhenti dan memberi tahu.

> **Catatan tentang model ini.** Pra-pelatihannya mengajarkan suara itu apa, bukan suara itu asli atau buatan. Dua pertanyaan yang berbeda, dan itu terlihat pada hasilnya.

Dua notebook pendamping membandingkan pilihan pelatihan pada model yang sama:
`Optimizer_AST.ipynb` dan `Dropout_AST.ipynb`.

## Langkah 1 · Menyiapkan runtime, kode, dan dependensi

Yang terjadi di sel ini, berurutan:

1. **Memeriksa kartu grafisnya.** Bukan sekadar ada atau tidak, tetapi jenisnya,
   karena kartu lama dan kartu baru memakai format bilangan yang berbeda saat
   melatih. Ini berpengaruh pada hasil, jadi dicatat sejak awal.
2. **Mengambil kode penelitian** dari GitHub. Tidak ada kode yang ditempel di
   dalam notebook ini, semuanya berasal dari repositori yang sama dengan yang
   dipakai di komputer lokal. Itu syarat supaya angkanya bisa dibandingkan.
3. **Memasang pustaka** yang belum dibawa Colab.
4. **Menjalankan pengaman konfigurasi**, yaitu skrip yang menolak melanjutkan
   kalau ada pengaturan yang menyimpang dari acuan penelitian.

In [ ]:
# Notebook ini khusus satu model. Nilainya sengaja ditulis tetap, bukan
# dropdown, supaya tidak ada yang tergeser tanpa sengaja saat presentasi.
MODEL      = "ast"
BATCH      = 32          # batch yang dipakai di mesin lokal untuk model ini
AUGMENTASI = "codec"    # perbaikan kebocoran codec, lihat notebook Proses

import os, subprocess, sys

AKAR = "/content/general-ai"
if not os.path.exists(os.path.join(AKAR, ".git")):
    subprocess.run(["git", "clone", "--quiet", "https://github.com/Tristan-tech-ai/general-AI.git", AKAR], check=True)
os.chdir(AKAR)
sys.path.insert(0, AKAR)

from colab.siapkan import siapkan
siapkan(MODEL)

## Langkah 2 · Mengambil dan memverifikasi dataset

Dataset yang dipakai adalah **Fake-or-Real, potongan dua detik**: 17.870 berkas
suara, masing masing tepat dua detik, 16.000 sampel per detik, satu kanal.

Yang penting di sini bukan mengunduhnya, tapi **membuktikan datanya sama**.
Kalau dataset di Colab berbeda sedikit saja dari yang dipakai di komputer
lokal, seluruh perbandingan angka menjadi tidak berarti.

Karena itu ada dua lapis pemeriksaan:

1. **Sidik jari arsip.** Kode sha256 dari berkas arsipnya dibandingkan dengan
   yang tercatat di repositori.
2. **Sidik jari pohon berkas.** Ini yang lebih dalam: setiap berkas wav
   diperiksa satu per satu, lalu diringkas menjadi satu kode. Pemeriksaan ini
   tidak peduli berkasnya datang dari mana atau namanya di folder apa. Yang
   dibandingkan isinya.

Kalau salah satu tidak cocok, sel ini **berhenti** dan tidak melanjutkan ke
pelatihan. Itu memang disengaja.

In [ ]:
from colab.siapkan import siapkan_dataset, verifikasi_dan_manifest

siapkan_dataset(buat_cache=True)
baris = verifikasi_dan_manifest()

## Langkah 3 · Mendengarkan dan melihat datanya

Sebelum membicarakan model, ada baiknya mendengar dulu apa yang harus
dibedakan. Sel ini mengambil satu suara asli dan satu suara palsu dari **data
uji**, memutarnya, lalu menggambarnya dengan dua cara.

Yang biasanya terjadi saat dipresentasikan: pendengar tidak bisa membedakan
keduanya, dan gambarnya pun terlihat mirip. Itu justru poinnya. Kalau
perbedaannya kentara, tidak perlu penelitian ini.

Dua gambar yang ditampilkan:

- **Gelombang**, yaitu tinggi sinyal terhadap waktu. Ini bentuk aslinya.
- **Spektrum**, yaitu peta energi tiap frekuensi di tiap saat. Warna terang
  berarti energi besar. Bagian atas gambar adalah frekuensi tinggi, dan di
  sanalah jejak mesin biasanya berada.

In [ ]:
import numpy as np, soundfile as sf, matplotlib.pyplot as plt
from scipy.signal import stft
from IPython.display import Audio, display
from forlib.data import load_manifest

rows = load_manifest("manifest.csv")
uji = [r for r in rows if r["split_official"] == "testing"]
asli  = [r for r in uji if r["label"] == 0][7]
palsu = [r for r in uji if r["label"] == 1][7]

def baca(r):
    x, sr = sf.read(r["path"], dtype="float32")
    return (x.mean(axis=1) if x.ndim > 1 else x), sr

x_asli, SR = baca(asli)
x_palsu, _ = baca(palsu)

print(f"ASLI  : {asli['fname']}")
display(Audio(x_asli, rate=SR))
print(f"PALSU : {palsu['fname']}")
display(Audio(x_palsu, rate=SR))

def gambar_spek(x):
    f, t, Z = stft(x, fs=SR, nperseg=512, noverlap=384)
    return f, t, 20 * np.log10(np.abs(Z) + 1e-8)

fig, ax = plt.subplots(2, 2, figsize=(13, 5.6), layout="constrained")
for kol, (x, judul_k, warna) in enumerate(
        [(x_asli, "ASLI", "#1864AB"), (x_palsu, "PALSU", "#C2255C")]):
    ax[0, kol].plot(np.arange(len(x)) / SR, x, lw=0.4, color=warna)
    ax[0, kol].set_title(f"{judul_k} - gelombang")
    ax[0, kol].set_xlabel("detik"); ax[0, kol].set_ylim(-1, 1)
    f, t, S = gambar_spek(x)
    ax[1, kol].imshow(S, aspect="auto", origin="lower", cmap="magma",
                      extent=[0, len(x) / SR, 0, SR / 2000],
                      vmin=S.max() - 80, vmax=S.max(), interpolation="nearest")
    ax[1, kol].set_title(f"{judul_k} - spektrum")
    ax[1, kol].set_xlabel("detik"); ax[1, kol].set_ylabel("kHz")
plt.show()

print("Kalau keduanya terdengar dan terlihat mirip, itu memang yang diharapkan.")
print("Perbedaannya ada, tapi terlalu halus untuk telinga dan mata manusia.")

## Langkah 4 · Apa yang sebenarnya dilihat AST

AST bekerja pada **gambar suara**, bukan gelombangnya. Urutannya begini:

1. Gelombang 32.000 angka diubah menjadi spektrum, yaitu peta energi tiap
   frekuensi di tiap saat.
2. Frekuensinya dipadatkan ke 128 pita mengikuti **skala Mel**, yaitu skala
   yang meniru cara telinga manusia mendengar.
3. Hasilnya dipotong menjadi petak petak kecil, dan tiap petak diperlakukan
   seperti satu kata dalam kalimat oleh Transformer.

Langkah kedua adalah yang harus dijelaskan hati hati. Skala Mel memberi
**banyak** detail pada frekuensi rendah, tempat suara manusia berada, dan
**sedikit** detail pada frekuensi tinggi. Padahal jejak mesin justru banyak
berada di frekuensi tinggi.

Sel di bawah menghitung sendiri seberapa besar pemadatan itu, langsung dari
definisi skala Mel yang dipakai model ini. Angkanya bukan kutipan, tapi hasil
hitungan di depan mata.

In [ ]:
import torch, numpy as np
from forlib.models import build_model

model = build_model("ast", freeze=True, layer_weighting=True).eval().cuda()

wav = torch.from_numpy(x_asli[:32000].astype("float32"))[None].cuda()
with torch.no_grad():
    fb = model._fbank(wav)

print(f"masukan model : {tuple(wav.shape)}   gelombang mentah")
print(f"setelah fbank : {tuple(fb.shape)}")
print(f"                = 1 klip, {fb.shape[1]} bingkai waktu, {fb.shape[2]} pita mel")
print()

cfg = model.encoder.config
f_grid, t_grid = model._grid(cfg)
print(f"petak         : {cfg.patch_size} x {cfg.patch_size}, "
      f"langkah {cfg.frequency_stride} x {cfg.time_stride}")
print(f"jumlah petak  : {f_grid} x {t_grid} = {f_grid * t_grid} petak")
print("tiap petak diperlakukan seperti satu kata oleh Transformer\n")

# ---- Seberapa besar sebenarnya pemadatan skala Mel? Dihitung, bukan dikutip.
SR_ = 16000
n_mel = cfg.num_mel_bins
mel_maks = 2595.0 * np.log10(1.0 + (SR_ / 2) / 700.0)
tepi_mel = np.linspace(0.0, mel_maks, n_mel + 1)
tepi_hz = 700.0 * (10.0 ** (tepi_mel / 2595.0) - 1.0)
lebar_hz = np.diff(tepi_hz)

print(f"lebar 1 pita mel di frekuensi terendah : {lebar_hz[0]:6.1f} Hz")
print(f"lebar 1 pita mel di frekuensi tertinggi: {lebar_hz[-1]:6.1f} Hz")
print(f"rasio                                  : {lebar_hz[-1] / lebar_hz[0]:6.1f} kali lebih kasar")
print()
print("Artinya detail di frekuensi tinggi dipadatkan belasan kali lebih rapat")
print("daripada di frekuensi rendah. Jejak mesin banyak berada di sana.")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), layout="constrained")
ax[0].imshow(fb[0].float().cpu().numpy().T, aspect="auto", origin="lower",
             cmap="magma", interpolation="nearest")
ax[0].set_title(f"Yang masuk ke AST: {fb.shape[1]} bingkai x {fb.shape[2]} pita mel")
ax[0].set_xlabel("bingkai waktu"); ax[0].set_ylabel("pita mel")
for k in range(f_grid + 1):
    ax[0].axhline(k * cfg.frequency_stride, color="w", lw=0.4, alpha=0.5)
for k in range(0, t_grid + 1, 2):
    ax[0].axvline(k * cfg.time_stride, color="w", lw=0.4, alpha=0.5)

ax[1].plot(tepi_hz[:-1], lebar_hz, color="#C2255C", lw=2)
ax[1].set_title("Lebar tiap pita mel, dalam Hz")
ax[1].set_xlabel("frekuensi (Hz)"); ax[1].set_ylabel("lebar 1 pita (Hz)")
ax[1].grid(alpha=0.25)
ax[1].annotate(f"{lebar_hz[-1] / lebar_hz[0]:.1f}x lebih kasar\ndi frekuensi tinggi",
               xy=(tepi_hz[-2], lebar_hz[-1]), xytext=(1500, lebar_hz[-1] * 0.72),
               arrowprops=dict(arrowstyle="->", color="#495057"), fontsize=9)
plt.show()

## Langkah 5 · Bentuk modelnya dan berapa yang benar benar dilatih

AST membawa sekitar 86 juta parameter dari pra-pelatihan AudioSet, dan
**hampir semuanya dibekukan**. Yang dilatih hanya bagian pengambil keputusan
di ujungnya.

Alasannya sama seperti pada model swa-selia: data latihnya hanya 13.956 klip,
sekitar 7,8 jam. Melatih ulang puluhan juta parameter dengan data sesedikit itu
hampir pasti menghafal, bukan belajar.

In [ ]:
tot = sum(p.numel() for p in model.parameters())
lat = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total parameter : {tot:>12,}")
print(f"dilatih         : {lat:>12,}   ({lat / tot * 100:5.2f} persen)")
print(f"dibekukan       : {tot - lat:>12,}   ({(tot - lat) / tot * 100:5.2f} persen)")
print()
print(f"{'bagian':<16}{'parameter':>14}{'dilatih':>10}")
print("-" * 40)
for nama_bagian, modul in model.named_children():
    n = sum(p.numel() for p in modul.parameters())
    d = sum(p.numel() for p in modul.parameters() if p.requires_grad)
    if n:
        print(f"{nama_bagian:<16}{n:>14,}{'ya' if d else 'tidak':>10}")
print()
print("Jalur bentuk data dari gelombang sampai jawaban:\n")
with torch.no_grad():
    x = model._fbank(wav)
    print(f"  gelombang masuk        {tuple(wav.shape)}")
    print(f"  gambar suara (fbank)   {tuple(x.shape)}")
    h_all = model.encoder(x, output_hidden_states=True).hidden_states
    h = model.lw(h_all)
    print(f"  keluaran encoder       {len(h_all)} x {tuple(h_all[0].shape)}")
    print(f"  digabung pakai bobot   {tuple(h.shape)}")
    h = model.bottleneck(h.transpose(1, 2))
    print(f"  conv penyempit         {tuple(h.shape)}")
    z = model.pool(h)
    print(f"  attentive pooling      {tuple(z.shape)}")
    o = model.head(z)
    print(f"  kepala keputusan       {tuple(o.shape)}   <- 2 angka: asli, palsu")
    p_ = torch.softmax(o, dim=1)[0]
    print(f"\n  tebakan model saat ini : asli {p_[0]:.3f} / palsu {p_[1]:.3f}")
    print("  (masih asal asalan, model ini belum dilatih sama sekali)")

## Langkah 6 · Bagaimana data sengaja dirusak, dan kenapa

Ini bagian yang paling layak dijelaskan panjang, karena di sinilah letak
temuan yang tidak ada di rencana awal penelitian.

Waktu datanya diperiksa satu per satu, ketahuan ada masalah serius. Di **data
latih**, hampir semua suara palsu berasal dari berkas MP3, sedangkan di **data
uji** tidak ada satu pun. Sel di bawah menghitung angkanya langsung dari
datanya, jadi bukan kutipan.

Kenapa itu masalah. Kompresi MP3 memotong frekuensi tinggi. Kalau hampir semua
suara palsu di data latih terpotong frekuensi tingginya sedangkan suara aslinya
tidak, model tidak perlu belajar mengenali jejak mesin sama sekali. Cukup
belajar satu aturan pendek: **frekuensi tingginya hilang berarti palsu.**

Aturan itu bekerja sempurna di data latih dan **gagal total** di data uji,
karena di sana tidak ada satu pun yang berasal dari MP3.

Perbaikannya sederhana dan bisa dijelaskan dalam satu kalimat: **potong
frekuensi tinggi secara acak pada kedua kelas.** Suara asli juga dipotong,
suara palsu juga dipotong, jumlahnya acak. Setelah itu memotong frekuensi
tinggi tidak lagi menandakan apa apa, dan model terpaksa mencari jejak yang
sesungguhnya.

In [ ]:
from forlib.data import codec_augment

# ---- Berapa besar sebenarnya kebocorannya? Dihitung dari datanya sendiri.
tr_palsu = [r for r in rows if r["split_official"] == "training" and r["label"] == 1]
te_palsu = [r for r in rows if r["split_official"] == "testing" and r["label"] == 1]
p_tr = sum(r["is_mp3"] for r in tr_palsu) / len(tr_palsu) * 100
p_te = sum(r["is_mp3"] for r in te_palsu) / len(te_palsu) * 100
print(f"suara PALSU di data latih yang berasal dari MP3 : {p_tr:5.1f} persen  "
      f"({sum(r['is_mp3'] for r in tr_palsu)} dari {len(tr_palsu)})")
print(f"suara PALSU di data uji   yang berasal dari MP3 : {p_te:5.1f} persen  "
      f"({sum(r['is_mp3'] for r in te_palsu)} dari {len(te_palsu)})")
print()
print("Selisih sebesar itu adalah jalan pintas yang menganga. Model yang")
print("memakainya akan terlihat sangat pintar saat latihan lalu jatuh saat diuji.")
print()

rng = np.random.default_rng(0)
x_rusak = codec_augment(x_asli.astype(np.float64), rng).astype(np.float32)

print("Suara ASLI sebelum dirusak:")
display(Audio(x_asli, rate=SR))
print("Suara ASLI yang sama, setelah frekuensi tingginya dipotong acak:")
display(Audio(x_rusak, rate=SR))

def rerata_spektrum(x):
    f, _, Z = stft(x, fs=SR, nperseg=512, noverlap=384)
    return f, 20 * np.log10(np.abs(Z).mean(axis=1) + 1e-8)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), layout="constrained")
f, s0 = rerata_spektrum(x_asli)
_, s1 = rerata_spektrum(x_rusak)
ax[0].plot(f / 1000, s0, color="#1864AB", lw=1.4, label="sebelum")
ax[0].plot(f / 1000, s1, color="#E8590C", lw=1.4, label="setelah dipotong")
ax[0].set_title("Rerata energi tiap frekuensi")
ax[0].set_xlabel("kHz"); ax[0].set_ylabel("dB"); ax[0].legend(); ax[0].grid(alpha=0.25)

ax[1].bar(["latih", "uji"], [p_tr, p_te], color=["#C2255C", "#2F9E44"], width=0.5)
ax[1].set_title("Persen suara PALSU yang berasal dari MP3")
ax[1].set_ylabel("persen"); ax[1].set_ylim(0, 100)
for i, v in enumerate([p_tr, p_te]):
    ax[1].text(i, v + 2, f"{v:.1f}%", ha="center", fontsize=11)
plt.show()

print("Titik potongnya diacak antara 3.000 dan 7.800 Hz, berbeda tiap berkas")
print("dan tiap putaran latihan, dan dikenakan pada kedua kelas tanpa pandang bulu.")

## Langkah 7 · Melatih

Sekarang barulah pelatihannya berjalan. Yang terjadi di tiap putaran:

1. Data latih diacak urutannya, lalu dibagi menjadi rombongan berisi
   32 klip.
2. Tiap klip dibaca dari berkas, dirusak secara acak seperti di langkah 6, lalu
   disamakan kerasnya.
3. Model menebak, tebakannya dibandingkan dengan jawaban benar, lalu bagian
   yang dilatih digeser sedikit ke arah yang lebih benar.
4. Sesudah satu putaran penuh, model diuji pada data validasi. Kalau hasilnya
   lebih baik dari putaran sebelumnya, bobotnya disimpan.

Yang disimpan pada akhirnya adalah **putaran terbaik menurut data validasi**,
bukan putaran terakhir. Data uji tidak pernah dilihat selama pelatihan, satu
kali pun.

> **Waktu.** Pada kartu T4 gratis, sepuluh putaran biasanya belasan sampai dua puluhan menit. Untuk memperlihatkan prosesnya saja, turunkan
> `EPOCHS` menjadi 2 atau 3.

Hasilnya ditulis ke `runs_colab/`, sengaja terpisah dari `runs/` yang berisi
hasil komputer lokal. Keduanya tidak boleh dicampur, karena kartu grafis dan
format bilangannya berbeda, dan selisih akibat perangkat keras akan terbaca
seolah olah selisih akibat inisialisasi acak.

In [ ]:
#@title Setelan pelatihan { display-mode: "form" }
EPOCHS = 10 #@param {type:"slider", min:1, max:20, step:1}
SEED = 42 #@param {type:"integer"}
SIMPAN_HASIL_KE_DRIVE = False #@param {type:"boolean"}

import time
from colab.siapkan import jalankan, berhenti

# Tag dibentuk dengan aturan yang sama seperti train.py, sehingga langkah 8
# tahu persis folder mana yang harus dibaca tanpa menebak lewat pencarian.
TAG = f"{MODEL}_official_{AUGMENTASI}_b{BATCH}e{EPOCHS}_s{SEED}"

print(f"model      : {MODEL}")
print(f"augmentasi : {AUGMENTASI}")
print(f"epoch      : {EPOCHS}   batch: {BATCH}   seed: {SEED}")
print(f"keluaran   : runs_colab/{TAG}\n")

t0 = time.time()
kode, _ = jalankan([
    sys.executable, "train.py",
    "--model", MODEL, "--split", "official", "--augment", AUGMENTASI,
    "--epochs", str(EPOCHS), "--batch", str(BATCH), "--workers", "2",
    "--seed", str(SEED), "--out", "runs_colab",
])
if kode != 0:
    berhenti("Pelatihan gagal. Kalau pesannya menyebut kehabisan memori GPU, "
             "turunkan BATCH menjadi 8 lalu jalankan sel ini lagi.")
print(f"\npelatihan selesai dalam {(time.time() - t0) / 60:.1f} menit")

if SIMPAN_HASIL_KE_DRIVE:
    from colab.siapkan import simpan_ke_drive
    simpan_ke_drive()

## Langkah 8 · Membaca hasil dan kesalahannya

Tiga hal yang dibaca di sini, dan urutannya penting.

**Ambang keputusan.** Model mengeluarkan angka antara 0 dan 1, bukan jawaban
ya atau tidak. Angka itu harus dipotong di suatu titik. Memotong di 0,5 terasa
wajar tetapi sebenarnya sewenang wenang. Sel ini melaporkan dua duanya: pada
ambang 0,5, dan pada ambang yang disesuaikan dengan perbandingan kelas di data
uji. Selisih keduanya adalah ukuran langsung seberapa meleset kalibrasinya.

**AUC dan EER.** Dua ukuran ini tidak bergantung pada ambang sama sekali. Kalau
akurasinya jelek tetapi AUC-nya tinggi, artinya model sebenarnya bisa
membedakan, hanya salah menaruh garis potongnya. Itu masalah yang jauh lebih
ringan, dan membedakan keduanya adalah salah satu temuan penelitian ini.

**Kesalahan yang paling percaya diri.** Sel ini memutar berkas yang salah
ditebak model dengan keyakinan paling tinggi. Ini bagian yang biasanya paling
menarik saat presentasi, karena orang bisa mendengar sendiri di mana modelnya
tertipu.

**Bobot lapisan.** Encoder mengeluarkan belasan lapisan sekaligus, dan model
mempelajari sendiri lapisan mana yang berguna. Grafik batangnya ditampilkan di
bawah. Kalau bobotnya menumpuk di lapisan tengah, itu bacaan yang menarik:
lapisan paling akhir dipra-latih untuk tugas lain, sedangkan lapisan tengah
menyimpan lebih banyak ciri akustik mentah.

In [ ]:
import json
import numpy as np
from forlib.metrics import full_metrics, prior_matched_threshold

d = f"runs_colab/{TAG}"
if not os.path.exists(f"{d}/results.json"):
    raise SystemExit(f"Belum ada hasil di {d}. Jalankan langkah 7 lebih dahulu.")

y, p, _ = np.load(f"{d}/test_scores.npy")
y = y.astype(int)
res = json.load(open(f"{d}/results.json"))

m05 = full_metrics(y, p, 0.5)
mpm = full_metrics(y, p, prior_matched_threshold(p, 0.5))

print(f"hasil dari : {d}")
print(f"putaran terbaik menurut validasi : epoch {res['best_epoch']}\n")
print(f"{'':<26}{'ambang 0,5':>13}{'ambang disesuaikan':>21}")
print("-" * 60)
for nama_m, k in [("akurasi", "accuracy"), ("recall (palsu terdeteksi)", "recall"),
                  ("spesifisitas (asli aman)", "specificity"), ("F1", "f1")]:
    print(f"{nama_m:<26}{m05[k] * 100:>12.2f}%{mpm[k] * 100:>20.2f}%")
print("-" * 60)
print(f"{'AUC':<26}{m05['auc']:>13.4f}   <- tidak bergantung ambang")
print(f"{'EER':<26}{m05['eer'] * 100:>12.2f}%   <- tidak bergantung ambang")
print()
selisih = (mpm["accuracy"] - m05["accuracy"]) * 100
print(f"Akurasi yang hilang semata karena ambangnya meleset: {selisih:+.2f} poin.")
print("Kalau angka ini besar sedangkan AUC tinggi, yang gagal adalah")
print("kalibrasinya, bukan kemampuan modelnya membedakan.")

if "layer_weights" in res:
    w = np.array(res["layer_weights"])
    fig, ax = plt.subplots(figsize=(9, 2.8), layout="constrained")
    ax.bar(range(len(w)), w, color="#1864AB")
    ax.set_title("Bobot yang dipelajari untuk tiap lapisan encoder")
    ax.set_xlabel("lapisan (0 = embedding, terakhir = paling atas)")
    ax.set_ylabel("bobot")
    ax.axhline(1 / len(w), color="#868E96", ls="--", lw=1)
    ax.text(0.2, 1 / len(w) * 1.06, "garis putus putus = kalau semua sama rata",
            fontsize=8, color="#495057")
    plt.show()
    print(f"Lapisan paling berguna menurut model: lapis {int(w.argmax())} "
          f"dari {len(w) - 1}.")

te_rows = [r for r in rows if r["split_official"] == "testing"]
amb = prior_matched_threshold(p, 0.5)
tebak = (p >= amb).astype(int)
salah = np.where(tebak != y)[0]
print(f"\nsalah tebak: {len(salah)} dari {len(y)} berkas "
      f"({len(salah) / len(y) * 100:.2f} persen)")

if len(te_rows) != len(y):
    print(f"\n(daftar berkas {len(te_rows)} tidak sepadan dengan skor {len(y)}, "
          f"pemutaran contoh kesalahan dilewati)")
elif len(salah):
    for i in salah[np.argsort(-np.abs(p[salah] - amb))][:3]:
        r = te_rows[i]
        print(f"\n  {r['fname']}")
        print(f"  sebenarnya {'ASLI' if y[i] == 0 else 'PALSU'}, "
              f"ditebak {'PALSU' if tebak[i] == 1 else 'ASLI'}, "
              f"skor {p[i]:.4f} (ambang {amb:.4f})")
        x, sr = sf.read(r["path"], dtype="float32")
        display(Audio(x.mean(axis=1) if x.ndim > 1 else x, rate=sr))

## Penutup

Angka yang keluar dari notebook ini berasal dari **satu inisialisasi acak**.
Itu belum cukup untuk menyimpulkan apa pun.

Di komputer lokal, AST (Audio Spectrogram Transformer) pada konfigurasi yang sama menghasilkan
**86,43 persen (simpangan 2,94 atas tiga inisialisasi)**. Simpangan itu bukan hiasan. Ia berarti kalau notebook
ini dijalankan ulang dengan `SEED` yang berbeda, hasilnya akan bergeser
beberapa poin, padahal tidak ada satu pun hal lain yang berubah.

Karena itu, sebelum membandingkan model ini dengan Wav2Vec2, HuBERT, atau
CNN-LSTM, jalankan langkah 7 sekali lagi dengan `SEED = 1337` lalu sekali
lagi dengan `SEED = 2024`. Baru dari tiga angka itu perbandingannya layak
dibicarakan.

Dua belas notebook di rangkaian ini:

| model | proses | optimizer | dropout |
|---|---|---|---|
| Wav2Vec2 | `Proses_Wav2Vec2.ipynb` | `Optimizer_Wav2Vec2.ipynb` | `Dropout_Wav2Vec2.ipynb` |
| AST (Audio Spectrogram Transformer) | `Proses_AST.ipynb` | `Optimizer_AST.ipynb` | `Dropout_AST.ipynb` |
| HuBERT | `Proses_HuBERT.ipynb` | `Optimizer_HuBERT.ipynb` | `Dropout_HuBERT.ipynb` |
| CNN-LSTM | `Proses_CNN_LSTM.ipynb` | `Optimizer_CNN_LSTM.ipynb` | `Dropout_CNN_LSTM.ipynb` |